# Nuclear Segmentation Benchmark

This notebook wraps `benchmark_nuclear_segmentation.py` for interactive runs.

Workflow:

1. Update the batch configuration cell.
2. Add one entry per model you want to compare.
3. Run the benchmark cell once.
4. One CSV is written per model, and the summary plot updates from all CSV files in `model_benchmarking/results/`.


In [1]:
from pathlib import Path
import sys

from IPython.display import Image as NotebookImage, display

try:
    import pandas as pd
except ImportError:
    pd = None

cwd = Path.cwd().resolve()
if cwd.name == "model_benchmarking":
    repo_root = cwd.parent
else:
    repo_root = cwd

benchmark_root = repo_root / "model_benchmarking"
if not benchmark_root.exists():
    raise FileNotFoundError(
        f"Could not find model_benchmarking under {repo_root}. Start Jupyter from the repo root or the model_benchmarking folder."
    )

if str(benchmark_root) not in sys.path:
    sys.path.insert(0, str(benchmark_root))

from benchmark_nuclear_segmentation import run_benchmark, write_csv, write_summary_plot

benchmark_root


/Users/m298117/anaconda3/envs/senoquant-dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/m298117/Desktop/senoquant-dev/model_benchmarking')

In [2]:
model_specs = [
    # {
    #     "name": "cpsam",
    #     "models_root": None,
    #     "settings": {},
    # },
    {
        "name": "default_2d",
        "models_root": None,
        "settings": {},
    },
    # {
    #     "name": "my_external_model",
    #     "models_root": benchmark_root / "models",
    #     "settings": {},
    # },
]

images_dir = benchmark_root / "images"
ground_truth_dir = benchmark_root / "ground_truth"
plot_path = benchmark_root / "results" / "benchmark_summary.png"
plot_title = images_dir.name

print("Models:")
for spec in model_specs:
    print(f"  - {spec['name']}")
print(f"Images: {images_dir}")
print(f"Ground truth: {ground_truth_dir}")
print(f"Plot output: {plot_path}")


Models:
  - default_2d
Images: /Users/m298117/Desktop/senoquant-dev/model_benchmarking/images
Ground truth: /Users/m298117/Desktop/senoquant-dev/model_benchmarking/ground_truth
Plot output: /Users/m298117/Desktop/senoquant-dev/model_benchmarking/results/benchmark_summary.png


In [3]:
all_results = {}

for spec in model_specs:
    model_name = spec["name"]
    models_root = spec.get("models_root")
    settings = spec.get("settings", {})
    output_csv = benchmark_root / "results" / f"{model_name}.csv"

    rows = run_benchmark(
        model_name=model_name,
        images_dir=images_dir,
        ground_truth_dir=ground_truth_dir,
        settings=settings,
        models_root=models_root,
    )
    write_csv(output_csv, rows)
    all_results[model_name] = rows
    print(f"Wrote {output_csv}")

write_summary_plot(
    csv_dir=benchmark_root / "results",
    plot_path=plot_path,
    title=plot_title,
)

mean_rows = [
    next(row for row in rows if row["case_id"] == "MEAN")
    for rows in all_results.values()
]

if pd is not None:
    display(pd.DataFrame(mean_rows))
else:
    mean_rows

display(NotebookImage(filename=str(plot_path)))
print(f"Wrote {plot_path}")


Benchmarking default_2d:   0%|          | 1/1324 [00:57<21:14:19, 57.79s/image]


KeyboardInterrupt: 